In [2]:
import json
import re
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, List, Any

# --- CONFIGURATION ---
# Update these to point to your JSON output folders
# Example: Path("graph_files_iterative/A_bat_and_a_ball...")
SUBJECT_DIR = Path("../graph_files_iterative/A_bat_and_a_ball_cost_110_in_total_The_bat_cost") 
CONTROL_DIR = Path("../graph_files_iterative/I_have_some_apples_If_I_buy_2_more_I_will_have_5")

# Filter out early layers (0-5) that just process syntax
MIN_LAYER = 5
PERSISTENCE_THRESHOLD = 0.5

In [3]:
def extract_step_number(filename: str) -> int:
    """Extracts '12' from 'step_12_answer.json'"""
    match = re.search(r"step_(\d+)_", filename)
    return int(match.group(1)) if match else -1

def load_graphs(directory: Path) -> Dict[int, Any]:
    """Loads all .json graph files from a directory."""
    if not directory.exists():
        print(f"⚠️ Directory not found: {directory}")
        return {}
        
    # Look for .json files
    graph_files = list(directory.glob("*.json"))
    graph_files.sort(key=lambda p: extract_step_number(p.name))
    
    loaded_data = {}
    print(f"📂 Loading {len(graph_files)} steps from: {directory.name}...")
    
    for filepath in graph_files:
        step_num = extract_step_number(filepath.name)
        if step_num == -1: continue
            
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                loaded_data[step_num] = data
        except Exception as e:
            print(f"   ❌ Error reading {filepath.name}: {e}")
            
    print(f"   ✅ Loaded {len(loaded_data)} steps.")
    return loaded_data

In [ ]:
def track_persistence(graphs_dict: Dict[int, Any], min_layer: int) -> Dict[str, float]:
    """
    Tracks how many unique steps each feature appears in.
    Returns: Dict { "L{layer}.F{feature}": persistence_score }
    """
    feature_counts = defaultdict(set)
    total_steps = len(graphs_dict)
    
    if total_steps == 0: return {}

    for step_num, graph in graphs_dict.items():
        # In the JSON format, nodes are usually under a 'nodes' key
        nodes = graph.get('nodes', [])
        
        for node in nodes:
            # Extract ID info (adjust keys if your JSON uses different names)
            layer = node.get('layer', -1)
            feature = node.get('feature', -1)
            node_id = node.get('node_id', -1).split('_')
            # node_id = [int(x) for x in node.get('node_id',-1).split('_')]
            index= int(node_id[1])
            
            
            # FILTERS
            if feature == -1: continue      # Ignore reconstruction errors
            try:
                if int(layer) < min_layer: continue # Ignore input layers
            except:
                continue # Skip if layer isn't a number
            
            # Create ID
            unique_id = f"L{layer}.F{index}"
            feature_counts[unique_id].add(step_num)
            
    # Calculate scores
    return {fid: len(steps) / total_steps for fid, steps in feature_counts.items()}

def get_high_persistence(scores: Dict[str, float], threshold: float) -> Set[str]:
    return {fid for fid, score in scores.items() if score >= threshold}

In [7]:
# Load one of your subject graphs (e.g., Step 5)
import json
from pathlib import Path

# Replace with one of your actual file paths
test_file = list(SUBJECT_DIR.glob("*.json"))[1] 

with open(test_file, 'r') as f:
    data = json.load(f)

# Find a Transcoder node (usually Layer 5+)
for node in data['nodes']:
    if int(node.get('layer', -1)) >= 5:
        print("\n--- FOUND TRANSCODER NODE ---")
        print(json.dumps(node, indent=2))
        break


--- FOUND TRANSCODER NODE ---
{
  "node_id": "5_3992_1",
  "feature": 7993995,
  "layer": "5",
  "ctx_idx": 1,
  "feature_type": "cross layer transcoder",
  "token_prob": 0.0,
  "is_target_logit": false,
  "run_idx": 0,
  "reverse_ctx_idx": 0,
  "jsNodeId": "5_3992-0",
  "clerp": "",
  "influence": 0.41780218482017517,
  "activation": 27.0
}


In [8]:
# 1. Load Data
print("--- Loading Subject (Bat & Ball) ---")
subject_graphs = load_graphs(SUBJECT_DIR)

print("\n--- Loading Control (Apples) ---")
control_graphs = load_graphs(CONTROL_DIR)

# 2. Track Persistence
subject_scores = track_persistence(subject_graphs, MIN_LAYER)
control_scores = track_persistence(control_graphs, MIN_LAYER)

# 3. Filter
subject_set = get_high_persistence(subject_scores, PERSISTENCE_THRESHOLD)
control_set = get_high_persistence(control_scores, PERSISTENCE_THRESHOLD)

print(f"\nPersistent Features found:")
print(f"  Subject: {len(subject_set)}")
print(f"  Control: {len(control_set)}")

--- Loading Subject (Bat & Ball) ---
📂 Loading 21 steps from: A_bat_and_a_ball_cost_110_in_total_The_bat_cost...
   ✅ Loaded 20 steps.

--- Loading Control (Apples) ---
📂 Loading 19 steps from: I_have_some_apples_If_I_buy_2_more_I_will_have_5...
   ✅ Loaded 18 steps.


ValueError: invalid literal for int() with base 10: 'E'

In [ ]:
# Find features common to BOTH tasks
shared_circuit = subject_set.intersection(control_set)

print(f"\n{'='*40}")
print(f"🔍 SHARED MATH CIRCUIT FOUND")
print(f"{'='*40}")
print(f"Total Shared Features: {len(shared_circuit)}")

# Sort and display
sorted_features = sorted(list(shared_circuit), key=lambda x: int(x.split('.')[0].replace('L','')))

print(f"\n{'Feature ID':<15} | {'Bat %':<10} | {'Apples %':<10}")
print("-" * 40)

for fid in sorted_features[:20]: # Show top 20
    s = subject_scores[fid]
    c = control_scores[fid]
    print(f"{fid:<15} | {s:.0%}      | {c:.0%}")
# L5.F28091254    | 70%      | 100%


🔍 SHARED MATH CIRCUIT FOUND
Total Shared Features: 63

Feature ID      | Bat %      | Apples %  
----------------------------------------
L5.F28091254    | 70%      | 100%
L5.F7993995     | 100%      | 100%
L5.F11103822    | 65%      | 67%
L5.F68825772    | 95%      | 67%
L5.F34159239    | 85%      | 72%
L5.F69755760    | 50%      | 50%
L5.F2584395     | 85%      | 78%
L6.F48733121    | 80%      | 78%
L6.F105683984   | 100%      | 100%
L6.F10231019    | 80%      | 100%
L6.F15487388    | 85%      | 61%
L6.F106470521   | 100%      | 100%
L6.F2586668     | 100%      | 100%
L6.F94661913    | 60%      | 89%
L7.F2584393     | 55%      | 56%
L7.F126587908   | 50%      | 50%
L7.F5744347     | 85%      | 50%
L7.F110677      | 100%      | 100%
L7.F5740958     | 100%      | 78%
L7.F16528367    | 100%      | 100%
